# ¿Qué pueden hacer las CNN?

- Clasificación directa sobre imágenes
  - Tomar de entrada una imagen y devolver la etiqueta a predecir

- Detección de objetos
  - Se predice una "bounding box"
  - Se predice una clase que identifica el contenido de la "bounding box"

![LeNet-5](https://cdn.analyticsvidhya.com/wp-content/uploads/2024/07/134t-04-1.webp)

Para hacer lo anterior normalmente se predicen dos coordenadas (x,y) (regresión) y como son rectángulos se puede formar fácilmente la caja alrededor del objeto.

- Segmentación de objetos
  - Se realiza una perdicción sobre cada píxel.
  - Las etiquetas son máscaras precisas que rodean al objeto.

- Reconocimiento facial
  - Una forma es mediante el entrenamiento simultáneo de dos CNN, cuyas entradas serán una imagen de una persona y otra imagen de la misma persona pero idealmente distintas instancias.
  - Los resultados se comparan mediante una métrica de similaridad y se retropropaga la señal de entrenamiento.
  - Redes convolucionales siameses.

  ![](https://miro.medium.com/v2/resize:fit:1400/0*lgjFPlTjPjiW4ziu.png)


- Síntesis de imágenes
  - Generative Adversarial Networks
  - Se verán más adelante


- ¿Por qué no utilizamos simples MLP?
  - Una MLP asume que cada feature es independiente, en el caso de una imagen cada píxel es una característica
  - Capas completamente conectadas no tienen el poder de descifrar la relación entre un pixel y sus vecinos.
  - Transformaciones: traslación, cambio en contraste/luz, deformación o distorsiones
  - Las CNN explotan las relaciones de grupos de píxeles (locales)
  - Otras redes (RNN) explotan la secuencialidad
  - Las redes basadas en *transformers* (atencionales) son buenas para los dos casos anteriores, pudiendo aplicarse a dependencias y secuencias de largo tamaño

# Arquitecturas básicas de CNNs
### Capas
  - Capas convolucionales (extracción de características)
  - Capas de pooling (reducción de tamaño y agregación de características)
  - Capas totalmente conectadas (predicción)
### Funcionamiento básico

- A la imagen de entrada (blanco y negro, 1 canal) se le aplica un kernel o filtro en una forma "sliding window" para el ancho y el alto de la imagen
  - Este kernel genera un mapa de características
- Se repite el proceso para los N kernels de la entrada
- A la salida de una capa convolucional vamos a tener múltiples (N) mapas de características para una sola imagen de entrada
- La salida para las dimensiones alto y ancho tras aplicar un kernel es:

\begin{equation}
O_{\{h,w\}} = \frac{(\{h,w\} - |k| + 2\cdot |p|)}{|s|} + 1
\end{equation}

- Normalmente el término del tamaño del padding es 0. Esto puede causar pérdida de información en los bordes pero es aceptable.
  - Esto ocurre porque cuando el kernel llega al final, no alcanza a cubrir por completo el final.
- El tamaño del kernel es un número impar (1, 3, 5, 7, ..., etc).


![](https://media5.datahacker.rs/2018/11/06_04.png)

### LeNet-5
- Entrada de 32x32
- Capa de convolución (5x5) * 6
  - Se generan 6 mapas de características de 28x28
- Capa de pooling (2x2) / (average pooling)
  - Los 6 mapas de 28x28 se convierten en 6 mapas de 14x14
- Capa de convolución (5x5) * 16
  - Se generan 16 mapas de características de 10x10
- Capa de pooling (2x2) / (average pooling)
  - Los 16 mapas de 10x10 se convierten en 16 mapas de 5x5
- Capa completamente conectada
  - El resultado anterior es convertido en un vector de características
  - Pasa todo a dos capas ocultas de tamaño 120 y 84
  - La salida es 10 (en este caso dígitos van de 0 a 9)
![LeNet-5](https://miro.medium.com/v2/resize:fit:1400/1*4Z6p3uo07w4GcZ_3QSmwPA.png)


### Otros conceptos

- Conexiones dispersas (sparse-connectivity)
  - Significa que un elemento en el mapa de características resultante está calculado mediante una región más pequeña (un parche) de la imagen.
  - A diferencia de un MLP que una sola neurona se conecta directamente a todos los pixeles, y así, consecutivamente.

- Compartimiento de parámetros
  - Los mismos pesos de los kernels se utilizan en diferentes regiones de la imagen
  - La intuición detrás de esto es que si logramos obtener un kernel que detecta una característica importante (por ej., bordes), también ayudará a detectar bordes en otras secciones de la imagen
  - Parámetros/pesos se reducen drásticamente

- Construcción sobre patrones
  - Se inicia obteniendo características locales para extraer características a nivel global

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
train_images, test_images = train_images / 255.0, test_images / 255.0

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
print(train_images[0].shape)

(32, 32, 3)


In [ ]:
train_images_grayscale = tf.image.rgb_to_grayscale(train_images)
test_images_grayscale = tf.image.rgb_to_grayscale(test_images)
print(train_images_grayscale[0].shape)

(32, 32, 1)


### Uso de parámetros

Una red neuronal con una capa oculta para una imagen (32, 32) tendría como entrada un vector de tamaño = 1024. Si cada entrada se conecta a una neurona en la capa oculta, con 10 unidades ocultas, tendriamos 1024*10 = 10 240 + 10 = 10 250  pesos.

Para ver las diferencias en eficiencia con una capa convolucional, usaremos la API Sequential de tensorflow/keras. La capa más común es **Conv2D** que agrupa varios filtros del mismo tamaño.


In [ ]:
model = models.Sequential([
    layers.Input((32, 32, 1)), # Notar que no es necesario usar "flatten"
    layers.Conv2D(1, (5, 5), activation="relu", kernel_initializer="glorot_uniform", strides=1, padding="valid")
])

model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 1)      │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26 (104.00 B)

 Trainable params: 26 (104.00 B)

 Non-trainable params: 0 (0.00 B)

En PyTorch haríamos lo siguiente:



Notamos que la cantidad de kernels se especificaría en "out_channels".

In [ ]:
import torch

conv_h = torch.nn.Conv2d(
  in_channels=1, out_channels=1,
  kernel_size=(5, 5),
)

print(conv_h.weight.size())
print(conv_h.bias.size())


torch.Size([1, 1, 5, 5])
torch.Size([1])


### Otras propiedades

- Tienen la propiedad de ser "shift invariant", es decir, un objeto en un lado de la imagen es igualmente reconocible por un filtro si se pone en cualquier otro lugar de la imagen.
- Esto por la compartición de parámetros y su procesamiento en forma "sliding window".
- No son invariantes ante transformaciones (de escala, rotación, traslación) por lo tanto el aumento de datos sirve a menudo para hacer más robustas las redes



### LeNet-5 en tf.keras


In [ ]:
# Definir arquitectura convolucional

model = models.Sequential([
  layers.Input((32, 32, 1)), # Entrada, notar que no es necesario usar "flatten" y especificamos los canales,

  # Extraer features
  layers.Conv2D(6, (5, 5), activation="relu"),  # Capa conv n°1
  layers.MaxPool2D(pool_size=(2, 2)),           # Pooling n°1
  layers.Conv2D(16, (5, 5), activation="relu"), # Capa conv n°2
  layers.MaxPool2D(pool_size=(2, 2)),           # Pooling n°1

  # Aplanar todo
  layers.Flatten(),

  # Classifier - MLP
  layers.Dense(120, activation="relu"),
  layers.Dense(84, activation="relu"),
  layers.Dense(10, activation="softmax"),
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
optim = tf.keras.optimizers.SGD() # learning_rate=..., momentum=..., etc...
model.compile(optimizer=optim, loss="sparse_categorical_crossentropy", metrics=["accuracy"])


In [ ]:
model.fit(train_images_grayscale, train_labels, epochs=20, batch_size=32)

Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - accuracy: 0.1432 - loss: 2.2575
Epoch 2/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - accuracy: 0.2762 - loss: 2.0076
Epoch 3/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 34s 21ms/step - accuracy: 0.3462 - loss: 1.8375
Epoch 4/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 32s 21ms/step - accuracy: 0.3949 - loss: 1.7014
Epoch 5/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 42s 21ms/step - accuracy: 0.4216 - loss: 1.6274
Epoch 6/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 32s 20ms/step - accuracy: 0.4491 - loss: 1.5642
Epoch 7/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 41s 21ms/step - accuracy: 0.4654 - loss: 1.5022
Epoch 8/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - accuracy: 0.4844 - loss: 1.4494
Epoch 9/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - accuracy: 0.5029 - loss: 1.4095
Epoch 10/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 41s 21ms/step - accuracy: 0.5178 - loss: 1.3702
Epoch 11/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 32s 20ms/step - accuracy: 0.5292 - loss: 1.3292
Epoch 12

In [ ]:
model.evaluate(test_images_grayscale, test_labels, verbose=2)
#from sklearn.metrics import accuracy_score
# preds = model.predict(test_images_grayscale)
# preds = tf.argmax(preds, axis=1)
# print(accuracy_score(test_labels, preds))

313/313 - 2s - 7ms/step - accuracy: 0.5730 - loss: 1.2260


[1.2260022163391113, 0.5730000138282776]

# Cálculo de parámetros para una CNN

En MLP, el principal cálculo de parámetros a aprender yacía cada par de capas _Fully Connected_. En CNN, además de estas capas densas, agregamos los parámetros aprendidos en las capas convolucionales:

$$(k_{altura \: filtro} \times k_{ancho \: filtro} \times n_{canales \: entrada} + 1) \times n_{filtros}$$

Donde el 1 representa al bias introducido por cada filtro. **Recordemos que no se aprende ningún parámetro en las capas de _Pooling_, solo se modifica la dimensión de la entrada.**

De esta manera, para la arquitectura LeNet-5, el total de parámetros a aprender se calcula mediante el siguiente cómputo:

$\text{params}_{conv2d_{1}} = (5 \times 5 \times 1 + 1) \times 6 = 156$

$\text{params}_{conv2d_{2}} = (5 \times 5 \times 6 + 1) \times 16 = 2416$

$\text{params}_{dense} = (400 \times 120) + 120 = 48120$

$\text{params}_{dense1} = (120 \times 84) + 84 = 10164$

$\text{params}_{dense2} = (84 \times 10) + 10 = 850$

$\sum \text{params} = 156 + 2416 + 48120 + 10164 + 850 = 61706$ parámetros aprendibles.

Corroborar este resultado con método `summary()` del modelo previamente construido.